# Online Course Recommendation System — Model Deployment

**Project:** P683 — Online Course Recommendation System
**Stage:** Deployment (Milestone 4)

This notebook takes the trained recommendation engine (TF-IDF + Cosine Similarity,
built in `online_CR.ipynb`) and deploys it as an interactive **Streamlit web app**.

**What this notebook does, step by step:**
1. Rebuild the cleaned dataset and the recommendation model
2. Save the model as reusable files (`.pkl`) — this is called *serialization*
3. Write a small web app (`app.py`) that loads those files and takes user input
4. Launch the app and expose it with a public link using `ngrok` (works from Google Colab)


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import warnings
warnings.filterwarnings("ignore")


## 2. Load the Cleaned Dataset

If you're running this in **Google Colab**, first upload `online_course_recommendation_v2.xlsx`
(or mount your Drive) and update the path below.


In [ ]:
# Update this path to wherever the dataset is located in your environment
DATA_PATH = "online_course_recommendation_v2.xlsx"

df = pd.read_excel(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## 3. Rebuild the Recommendation Model

This repeats the same feature engineering and TF-IDF steps used during model building,
so this notebook can run independently and produce the exact same model.


In [ ]:
# Select the columns relevant to content-based recommendation
df_features = df[[
    'course_name',
    'instructor',
    'difficulty_level',
    'certification_offered',
    'study_material_available'
]]

# Remove duplicate courses so each course appears only once
unique_courses = df_features.drop_duplicates(subset='course_name').reset_index(drop=True)
print("Unique courses:", unique_courses.shape)

# Combine text fields into a single string per course
unique_courses['combined_features'] = (
    unique_courses['course_name'].astype(str) + ' ' +
    unique_courses['instructor'].astype(str) + ' ' +
    unique_courses['difficulty_level'].astype(str) + ' ' +
    unique_courses['certification_offered'].astype(str) + ' ' +
    unique_courses['study_material_available'].astype(str)
)

# Convert text into TF-IDF vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_courses['combined_features'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# Map course name -> row index for fast lookup
course_indices = pd.Series(unique_courses.index, index=unique_courses['course_name'])


In [ ]:
def recommend_courses(course_name, top_n=5):
    """Return the top_n courses most similar to course_name."""
    if course_name not in course_indices.index:
        return "Course not found"

    idx = course_indices[course_name]
    cosine_scores = linear_kernel(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()

    similar_courses = cosine_scores.argsort()[::-1]
    similar_courses = [i for i in similar_courses if i != idx][:top_n]

    return unique_courses.iloc[similar_courses][[
        'course_name', 'instructor', 'difficulty_level',
        'certification_offered', 'study_material_available'
    ]].reset_index(drop=True)

# Quick sanity check
recommend_courses("Python for Beginners")


## 4. Save the Model Artifacts (Serialization)

We save four files that the web app will load later, instead of retraining every time:
- `tfidf_vectorizer.pkl` — the fitted TF-IDF vectorizer
- `tfidf_matrix.pkl` — the TF-IDF matrix of all courses
- `course_data.pkl` — the cleaned, de-duplicated course table
- `course_indices.pkl` — lookup table from course name to row index


In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("course_data.pkl", "wb") as f:
    pickle.dump(unique_courses, f)

with open("course_indices.pkl", "wb") as f:
    pickle.dump(course_indices, f)

print("All model artifacts saved successfully.")


## 5. Build the Web App (Streamlit)

`%%writefile` saves the code below as a standalone file, `app.py`. This is the actual
application that end users will interact with — it loads the saved model files and
provides a simple UI: pick a course, get recommendations.


In [ ]:
%%writefile app.py
import pickle
import streamlit as st
from sklearn.metrics.pairwise import linear_kernel

st.set_page_config(page_title="Online Course Recommendation System", layout="centered")

@st.cache_resource
def load_artifacts():
    with open("tfidf_vectorizer.pkl", "rb") as f:
        tfidf = pickle.load(f)
    with open("tfidf_matrix.pkl", "rb") as f:
        tfidf_matrix = pickle.load(f)
    with open("course_data.pkl", "rb") as f:
        unique_courses = pickle.load(f)
    with open("course_indices.pkl", "rb") as f:
        course_indices = pickle.load(f)
    return tfidf, tfidf_matrix, unique_courses, course_indices

tfidf, tfidf_matrix, unique_courses, course_indices = load_artifacts()

def recommend_courses(course_name, top_n=5):
    if course_name not in course_indices.index:
        return None
    idx = course_indices[course_name]
    cosine_scores = linear_kernel(tfidf_matrix[idx:idx + 1], tfidf_matrix).flatten()
    similar_courses = cosine_scores.argsort()[::-1]
    similar_courses = [i for i in similar_courses if i != idx][:top_n]
    return unique_courses.iloc[similar_courses][
        ["course_name", "instructor", "difficulty_level",
         "certification_offered", "study_material_available"]
    ].reset_index(drop=True)

st.title("🎓 Online Course Recommendation System")
st.write("Select a course you like, and get similar course recommendations "
         "based on course content, instructor, difficulty level, and features.")

course_list = sorted(unique_courses["course_name"].unique())
selected_course = st.selectbox("Choose a course:", course_list)
top_n = st.slider("Number of recommendations:", min_value=1, max_value=10, value=5)

if st.button("Recommend"):
    results = recommend_courses(selected_course, top_n)
    if results is None:
        st.error("Course not found in the dataset.")
    else:
        st.subheader(f"Courses similar to: {selected_course}")
        st.dataframe(results, use_container_width=True)

st.markdown("---")
st.caption("Model: TF-IDF Vectorization + Cosine Similarity (Content-Based Filtering)")


## 6. Install Deployment Libraries

`streamlit` runs the web app. `pyngrok` creates a temporary public URL so the app
(running inside Colab/this notebook) can be opened in any browser.


In [ ]:
!pip install streamlit pyngrok -q


## 7. Launch the App

**One-time setup:** create a free account at https://dashboard.ngrok.com/signup,
copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken,
and paste it below in place of `"YOUR_NGROK_AUTHTOKEN"`.

Running this cell will:
1. Start the Streamlit app in the background
2. Open an `ngrok` tunnel to it
3. Print a public URL you can click to open the live app


In [ ]:
from pyngrok import ngrok
import time

# Paste your ngrok authtoken here (only needs to be run once per session)
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

# Kill any previous tunnels/streamlit processes, then start fresh
ngrok.kill()
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')
time.sleep(3)

public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)


## 8. (Optional) Stop the App

Run this when you're done demoing, to free up the port and close the tunnel.


In [ ]:
ngrok.disconnect(public_url)
ngrok.kill()


## Conclusion

The Online Course Recommendation System (TF-IDF + Cosine Similarity, content-based
filtering) has been successfully deployed as an interactive Streamlit web application.
Users can select any course and instantly receive similar course recommendations
through a simple, shareable web interface — completing the final milestone of the
project (Model Deployment).
